# Diffusion Tensor Imaging Measurements for Computer-Aided Diagnosis of Autism - Derived Data from ABIDE II Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The dataset contains DTI measurements for computer-aided diagnosis of autism, derived from ABIDE II.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

To programmatically inspect the record sets, fields, and columns in the dataset, we list them by their `@id` fields, as specified by the Croissant schema.

In [ ]:
# Retrieve all record sets by @id
record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record sets:")
record_set_ids = []
for rs in record_sets:
    print(f"  - Record set @id: {rs['@id']}, name: {rs.get('name', '<no name>')}")
    record_set_ids.append(rs['@id'])

# For each record set, list its fields and columns by @id
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    if fields:
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - Field @id: {f['@id']}, name: {f.get('name', '<no name>')}")
            else:
                print(f"    - Field @id: {f}")
    columns = rs.get('column', [])
    if not isinstance(columns, list):
        columns = [columns]
    if columns:
        print("  Columns:")
        for c in columns:
            if isinstance(c, dict):
                print(f"    - Column @id: {c['@id']}, name: {c.get('name', '<no name>')}")
            else:
                print(f"    - Column @id: {c}")

## 3. Data Extraction
Load data from the selected record set(s) into DataFrames for analysis.

Below, we dynamically extract each record set by its `@id`.

Choose record set(s) and field(s) for analysis from the overview above. All references are via their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id} with shape {df.shape}")

# Show column names for one record set
example_record_set = record_set_ids[0] if record_set_ids else None
if example_record_set:
    print("Columns (@id) in record set:")
    print(dataframes[example_record_set].columns.tolist())
    dataframes[example_record_set].head()
else:
    print("No record sets with records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All entities are referenced by their `@id` as per Croissant schema. Replace variable values depending on your analysis needs.

In [ ]:
# Example EDA on first record set
record_set_id = example_record_set  # Use record set @id from extraction
df = dataframes[record_set_id]

# Find a numeric field by inspecting columns
print("Available columns (@id):")
print(df.columns.tolist())
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64','float64']]
if not numeric_candidates:
    # Try to cast candidate columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notna().sum() > 0:
                numeric_candidates.append(col)
        except Exception:
            pass

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field for EDA (by @id): {numeric_field}")
else:
    print("No numeric fields detected.")

# Set threshold for filtering
threshold = 10

if numeric_candidates:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by another field
    group_candidates = [col for col in df.columns if col != numeric_field and df[col].nunique() < 10]
    group_field = group_candidates[0] if group_candidates else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, we use matplotlib for plotting histograms and scatterplots for the selected numeric field and group field.

In [ ]:
if example_record_set and numeric_candidates:
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Histogram of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot vs group field (if exists and is numeric)
    if group_field and df[group_field].dtype != 'O':
        plt.figure(figsize=(8,5))
        plt.scatter(df[group_field], df[numeric_field])
        plt.title(f"Scatter plot: {numeric_field} vs {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded metadata and records using `mlcroissant`.
- Inspected record sets, fields, and columns by their unique `@id`.
- Extracted and analyzed DTI metrics (such as fractional anisotropy) with basic filtering and normalization.
- Visualized numeric field distributions.
- This notebook serves as a FAIR, reproducible template for further exploration and machine learning tasks using the provided DTI dataset.